In [88]:
import os
import pandas as pd
from dotenv import load_dotenv #for API key hiding
import json
import time

In [89]:
df = pd.read_csv("national.csv")

In [90]:
df['state_id'].value_counts()

state_id
20    173
35    146
30    139
15     74
36     72
8      67
11     64
2      49
31     46
28     45
27     41
10     41
19     28
16     27
34     21
5      19
25     17
4      12
14      8
13      6
7       5
6       5
12      4
3       4
26      2
22      2
24      1
33      1
29      1
32      1
18      1
Name: count, dtype: int64

### Handcoding sample of 50 entries (~5% of the data)

In [185]:
# creating a random dataset of 50 complaints ~5% of all data. Commenting out after done so it doesn't keep running.
# handcoded = df.sample(50, random_state=42)
# what is random_state? Each number gives you a different random sample, but the same number gives you the same sample again.

In [5]:
# len(handcoded)

50

In [6]:
# # add groundtruth columns
# handcoded['human_animal_type'] = ""
# handcoded['human_dog_type'] = ""
# handcoded['human_complaint_type'] = ""

In [7]:
# # getting dataset into csv for handcoding
# handcoded.to_csv("handcoded_groundtruth_50_raw.csv", index=False)

In [91]:
# getting handcoded dataset back
handcoded = pd.read_csv("groundtruth_handcoded_50.csv")

In [92]:
# checking right things got imported
handcoded.head()

,sr_no,date_of_complaint,complaint_number,complainant_name_org,category_of_complaint,complaint,place_of_incident,violation_of_law,action_initiated,status,date_of_action_taken,state_id,page,human_animal_type,human_dog_type,human_complaint_type
0,13,26/01/2025,9-10/2024-25/1382/PCA,Unknown,-,Arrest that lady owner and make sure she suffe...,"Chetnanagar, Aurangabad, Maharashtra",-,-,-,05/08/2025,20,2,dog,stray,unclear_or_other
1,30,04/05/2025,9-7/2025-26/1749/PCA,Sourabh Bajaj,-,Humari gali me 10 se 15 dogs hai jo ki roj kis...,House no 775 sector 12,-,-,-,06/05/2025,11,1,dog,stray,harm_fear_or_nuisance_by_dogs
2,97,03/07/2024,9-4/2024-25/828/PCA,Rajeswari,-,Tribal peoples are killing and eating innocent...,https://maps.app.goo.gl/56uHdELB7hgQZowm8?g_st=ic,-,-,-,03/10/2024,30,1,dog,pet,NaN
3,45,08/04/2024,9-17/2024-25/564/PCA,MOHAMMAD SHA ABBAS,-,All Stray Dogs get bitten and murderd by peopl...,"Bhanu nagar,BRTS Road, Opposite Muthooth Finco...",-,-,-,10/07/2024,2,1,dog,stray,harm_or_threat_to_dogs
4,14,06/09/2025,9-13/2025-26/2311/PCA,ARNAB NAYAK,-,"Dear sir/Madam, \n Ne...","C/O- Amiya Kumar Nayak , VILL-BETGERIA\nP.O-KH...",-,-,-,22/10/2025,36,1,bird,NaN,NaN


In [93]:
# confirming there are no handcoded missing values for new columns
handcoded[
    ['human_animal_type', 'human_dog_type', 'human_complaint_type']
].isna().sum()
# pandas may have interpreted the string "NA" as a missing value NaN when reading the CSV. 
# For our comparison later, we want label to literally be "NA".

human_animal_type        0
human_dog_type          11
human_complaint_type    21
dtype: int64

In [94]:
# convert missing values to string "Not Applicable"
handcoded[['human_animal_type', 'human_dog_type', 'human_complaint_type']] = (
    handcoded[['human_animal_type', 'human_dog_type', 'human_complaint_type']]
    .fillna("Not Applicable")
)

In [95]:
# checking missing values again
handcoded[
    ['human_animal_type', 'human_dog_type', 'human_complaint_type']
].isna().sum()
# no missing values now!

human_animal_type       0
human_dog_type          0
human_complaint_type    0
dtype: int64

### Creating Prompt

In [161]:
# """ """ triple quote lets you write a long multi-line string. f makes it an f string.
# \"\"\"{complaint}\"\"\"  to differentiate complaint from the prompt.

def make_prompt(complaint):
    return f""" 
    
You are classifying animal welfare complaint allegations.
Classify based only on the text of the complaint. These are allegations, not verified facts.

Broader rules:
- Do not use outside knowledge. 
- Do not infer beyond what is written. 
- Choose exactly one value for each field.
- If multiple issues appear, choose the category that best captures the dominant allegation.
- Return only valid JSON.
- Do not include explanations, markdown, confidence scores, or extra keys.

Return this exact JSON structure:

{{
"animal_type": "",
"dog_type": "",
"complaint_type": ""
}}

Allowed values:

animal_type:

- dog 
- cow
- cat
- horse
- bird
- sheep
- unclear

dog_type:

- pet
- stray
- unclear
- Not Applicable

Allowed complaint_type values:
- harm_or_threat_to_dogs
- harm_or_threat_to_feeders
- harm_or_threat_to_dogs_and_feeders
- harm_due_to_fear
- harm_fear_or_nuisance_by_dogs
- unclear_or_other
- Not Applicable


1. animal_type:
- Use dog, cow, cat, horse, bird, sheep, unclear
- Use unclear if the complaint does not clearly mention the animal and/or if the animal mentioned is not in the allowed values.

2. dog_type:
- Applies only when animal_type is dog.
- pet: use when the complaint is about owned and/or companion dog.
- stray: use when the complaint is about stray dogs, street dogs, community dogs, free-roaming dogs, dogs in public spaces, or dogs cared for by feeders/caregivers. 
If the complaint mentions dog catchers, municipal capture, ABC pickup, sterilisation pickup, release, relocation, or dogs being taken from a public area, treat the dog as stray unless the complaint clearly says it is a pet dog.
Temporary care by residents, security guards, shelters, rescuers, or feeders does not make a dog a pet.
- unclear: complaint mentions dogs but does not make clear whether pet or stray/community.
- Not Applicable: use for all non-dog animal types.

3. complaint_type:

3a. Rules for complaint_type:

- This field only applies when animal_type is "dog" and dog_type is "stray"
- If animal_type is not "dog", complaint_type must be "Not Applicable".
- If dog_type is "pet", "unclear_or_other", or "Not Applicable", complaint_type must be "Not Applicable".

3b. Stray/community dog complaint_type categories:

* harm_or_threat_to_dogs:

Use when the complaint alleges stray/community dogs were harmed, removed, neglected, exploited, or threatened with harm by human beings.

This includes:
- physical cruelty, including poisoning, beating, abuse, injury, killing, or throwing harmful liquids
- threats to remove, relocate, poison, kill, injure, or otherwise harm stray/community dogs
- illegal relocation or removal of stray/community dogs
- dogs picked up and not returned to the same area
- dogs reported missing
- bad ABC, pound, or shelter conditions causing harm or neglect to dogs
- vehicle collision or hit-and-run cases involving stray/community dogs, whether accidental or intentional


* harm_or_threat_to_feeders:
Use when the main allegation is that feeders, caregivers, or people supporting stray/community dogs were harassed, threatened, abused, attacked, or obstructed because they feed, care for, or speak for stray/community dogs.
Use this only when the main allegation is harm or threat toward the feeder/caregiver, not direct harm or threat toward the dogs.
Do not use this category merely because a feeder is mentioned. Harm or threat of harm should be clear.


* harm_or_threat_to_dogs_and_feeders:

Use only when the complaint clearly alleges both:

1. Harm, threat, cruelty, or obstruction directed toward stray/community dogs; and
2. Harm, threat, intimidation, harassment, abuse, or obstruction directed toward a person feeding, watering, rescuing, caring for, or advocating for stray/community dogs.

Examples:
- "Residents threatened to poison the dogs and abused the feeder."
- "Residents beat the dogs and assaulted the person feeding them."
- "People threw bricks at the dogs and threatened the complainant for giving them water."
- "Residents destroyed the dogs' water bowls and intimidated the person who was caring for them."
- "Don't feed the dog and relocate it to shelter"

Do not use this category if only the dogs or only the feeder/caregiver is targeted. In those cases, use the corresponding single-target category.

harm_fear_or_nuisance_by_dogs:
Use when the complaint describes harm, fear, aggression, bites, barking, nuisance, danger, or requests to remove dogs because of public safety concerns.
This includes complaints where dogs are described as attacking, biting, chasing, barking, creating nuisance, or causing fear among residents.

harm_due_to_fear:
Use when the complaint alleges that stray/community dogs or feeders/caregivers were harmed, threatened, abused, removed, or obstructed as a reaction to bites and aggression and public safetly concerns involving dogs.
This category should be used when the complaint contains both:
a fear/harm/dangerous concerns involving dogs, and
harm, threat, abuse, removal, or retaliation directed toward dogs and/or feeders.

Only classify as harm_fear_or_nuisance_by_dogs when the complaint explicitly states or clearly alleges that the dog was harmed or threatened because people feared the dog or considered it dangerous or a nuisance due to bites or aggresion. Do not infer this motivation simply because people wanted dogs removed, banned, relocated, or poisoned.

Examples:
“The dog bit a child, so residents poisoned the dog.”
“After complaints about barking, residents beat the street dogs.”
“Residents are threatening the feeder because the dogs allegedly chase people.”
“Because of dog-bite fear, residents are demanding that the dogs be illegally relocated.”
Do not use this category when the complaint only says dogs are biting, barking, chasing, or creating nuisance. In that case, use "harm_fear_or_nuisance_by_dogs".

unclear_or_other:
Use when the stray/community dog complaint is too vague or does not fit the categories above.
E.g., "Stray dog was bitten / attacked by a pet dog", "Stray dog was attacked by a cow"

Complaint text:
\"\"\"{complaint}\"\"\" 
"""

### Running LLM

In [151]:
# getting API key
load_dotenv(".env", override=True)
api_key = os.getenv("api_key")

In [67]:
#getting package openai
%pip install openai

Note: you may need to restart the kernel to use updated packages.


In [152]:
from openai import OpenAI #OpenRouter uses the OpenAI Python package as a client. code means use the openai Python library to talk to an OpenAI-compatible API.
import os

client = OpenAI(
    api_key=os.getenv("api_key"),
    base_url="https://openrouter.ai/api/v1" #open router's URL
)

#### CLASSIFICATION PILOT

In [100]:
# # piloting one model 
# def ask_ai_to_classify(complaint):
#     prompt = make_prompt(complaint) #passes prompts

#     response = client.chat.completions.create( #This sends a request to the AI model. client is waiter, .chat.completions.create() = "take my order.", open Router is your restaurant.
#         model="openai/gpt-4o-mini",
#         messages=[
#             {"role": "user", "content": prompt} #The user is sending this prompt to the model.
#         ],
#         temperature=0 #This tells the model to be less creative and more consistent. temp = 0 means be as deterministic / rule-following as possible.
#     )

#     return response.choices[0].message.content #The full response contains lots of metadata. The actual AI answer is buried inside this.

In [71]:
# checking handcoded rows
handcoded.tail()

,sr_no,date_of_complaint,complaint_number,complainant_name_org,category_of_complaint,complaint,place_of_incident,violation_of_law,action_initiated,status,date_of_action_taken,state_id,page,human_animal_type,human_dog_type,human_complaint_type
45,87,01/04/2025,9-3/2025-26/1596/PCA,A Group of Residents,-,"Sir,\nSomeone has run over a car on a puppy wi...",Akansha Parisar Pocket A Jankipuram Sector F,-,-,-,01/04/2025,35,1,dog,stray,harm_or_threat_to_dogs
46,90,29/03/2025,9-3/2024-25/1578/PCA,Vikesh Rathi,-,The above person has kept a dog named Pitbull ...,Vpo sujti tehsil baraut,-,-,-,01/04/2025,35,1,dog,pet,Not Applicable
47,40,06/08/2024,9-10/2024-25/913/PCA,shabudin salim shaikh,-,"6th August, 2024\nAround 5: 30 pm, A cab Drive...",-,-,-,-,03/10/2024,20,2,dog,stray,harm_or_threat_to_dogs
48,6,05/11/2025,9-13/2025-26/2539/PCA,kuheli barari,-,This is an urgent plea to save a suffering cow...,"Bhibhutibhusan bandhopadhay sarani, 6a palm pl...",-,-,-,12/11/2025,36,1,cow,Not Applicable,Not Applicable
49,64,09/07/2025,9-3/2025-26/2020/PCA,-,-,"To,\nThe Chairperson Animal Welfare Board of I...",-,-,-,-,16/07/2025,35,1,dog,pet,Not Applicable


In [36]:
#getting first complaint 
handcoded.iloc[0]['complaint']

'Arrest that lady owner and make sure she suffer from the same pain by which that street dear dog died just bcz of thier carelessness and adopted pitbull dog. They are responsible for that and that poor street mother dog had child make sure also that her child should be safe and that pitbull should be at the animal jail/locker. Who know today he killed street dog tomorrow he can kill human children. These dog breeds are banned in India and still people are keeping them consider punishment for this also...and owner should be in jail. Give such examples that every owner who keeps these kind of dogs will remember 100 times and how shamefully she is abusing that person who tried to stop by throwing stone.'

In [54]:
# saving one complaint (im choosing second)
one_complaint = handcoded.iloc[1]['complaint']
one_complaint

'Humari gali me 10 se 15 dogs hai jo ki roj kisi na kisi ko kaat jate hai,\nHum gali wale sab bhut preshan hai. Humara ghr se bhar niklna mushkil hogya hai.\nAapse nivedan hai ki please inhe pakad ke khi or chhod aaye or hume inse chutkara dilwaye.\nChote bache bhi galli me khel nhi paate hai inki vjha se.\nPlease take an action on priority.'

In [73]:
# running one model on one complaint
ai_answer = ask_ai_to_classify(one_complaint)
print(ai_answer)
type(ai_answer)

{
"animal_type": "dog",
"dog_type": "stray",
"complaint_type": "harm_fear_or_nuisance_by_dogs"
}


str

In [79]:
# convering str into dictionary
ai_answer = json.loads(ai_answer)

TypeError: the JSON object must be str, bytes or bytearray, not dict

In [80]:
#adding to handcoded dataset 
idx = handcoded.index[1]

handcoded.loc[idx, "ai_animal_type"] = ai_answer["animal_type"]
handcoded.loc[idx, "ai_dog_type"] = ai_answer["dog_type"]
handcoded.loc[idx, "ai_complaint_type"] = ai_answer["complaint_type"]

In [87]:
# checking dataset
handcoded.head(2)

,sr_no,date_of_complaint,complaint_number,complainant_name_org,category_of_complaint,complaint,place_of_incident,violation_of_law,action_initiated,status,date_of_action_taken,state_id,page,human_animal_type,human_dog_type,human_complaint_type,ai_animal_type,ai_dog_type,ai_complaint_type
0,13,26/01/2025,9-10/2024-25/1382/PCA,Unknown,-,Arrest that lady owner and make sure she suffe...,"Chetnanagar, Aurangabad, Maharashtra",-,-,-,05/08/2025,20,2,dog,stray,unclear_or_other,NaN,NaN,NaN
1,30,04/05/2025,9-7/2025-26/1749/PCA,Sourabh Bajaj,-,Humari gali me 10 se 15 dogs hai jo ki roj kis...,House no 775 sector 12,-,-,-,06/05/2025,11,1,dog,stray,harm_fear_or_nuisance_by_dogs,dog,stray,harm_fear_or_nuisance_by_dogs


#### CLASSIFICATION FOR HANDCODED SAMPLE

##### ***Passing Prompt**

In [162]:
# AI classification function
def ask_ai_to_classify(complaint, model):
    prompt = make_prompt(complaint)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content

In [163]:
#choosing model 
model = "openai/gpt-4.1"

In [164]:
# itterating over each row of handcoded dataset and running prompt

for idx, row in handcoded.iterrows(): #if I write i for index, it might treat some other column as index. I am just being safe.
    complaint = row["complaint"] #get the complaint string out of each row by itterating one by one over each row

    try: #try so it doesn't stop on errors and goes on next row in case of error
        ai_answer = ask_ai_to_classify(complaint, model=model)
        parsed = json.loads(ai_answer)

        handcoded.loc[idx, "ai_animal_type"] = parsed["animal_type"]
        handcoded.loc[idx, "ai_dog_type"] = parsed["dog_type"]
        handcoded.loc[idx, "ai_complaint_type"] = parsed["complaint_type"]

        print(f"Done row {idx}")

    except Exception as e:
        print(f"Error on row {idx}: {e}")
        print(ai_answer if "ai_answer" in locals() else "No AI answer")

    time.sleep(1)

Done row 0
Done row 1
Done row 2
Done row 3
Done row 4
Done row 5
Done row 6
Done row 7
Done row 8
Done row 9
Done row 10
Done row 11
Done row 12
Done row 13
Done row 14
Done row 15
Done row 16
Done row 17
Done row 18
Done row 19
Done row 20
Done row 21
Done row 22
Done row 23
Done row 24
Done row 25
Done row 26
Done row 27
Done row 28
Done row 29
Done row 30
Done row 31
Done row 32
Done row 33
Done row 34
Done row 35
Done row 36
Done row 37
Done row 38
Done row 39
Done row 40
Done row 41
Done row 42
Done row 43
Done row 44
Done row 45
Done row 46
Done row 47
Done row 48
Done row 49


In [165]:
handcoded[
    [
        "complaint",
        "human_animal_type", "ai_animal_type",
        "human_dog_type", "ai_dog_type",
        "human_complaint_type", "ai_complaint_type"
    ]
].head()

,complaint,human_animal_type,ai_animal_type,human_dog_type,ai_dog_type,human_complaint_type,ai_complaint_type
0,Arrest that lady owner and make sure she suffer from the same pain by which that street dear dog died just bcz of thier carelessness and adopted pitbull dog. They are responsible for that and that poor street mother dog had child make sure also that her child should be safe and that pitbull should be at the animal jail/locker. Who know today he killed street dog tomorrow he can kill human children. These dog breeds are banned in India and still people are keeping them consider punishment for this also...and owner should be in jail. Give such examples that every owner who keeps these kind of dogs will remember 100 times and how shamefully she is abusing that person who tried to stop by throwing stone.,dog,dog,stray,stray,unclear_or_other,harm_or_threat_to_dogs
1,"Humari gali me 10 se 15 dogs hai jo ki roj kisi na kisi ko kaat jate hai,\nHum gali wale sab bhut preshan hai. Humara ghr se bhar niklna mushkil hogya hai.\nAapse nivedan hai ki please inhe pakad ke khi or chhod aaye or hume inse chutkara dilwaye.\nChote bache bhi galli me khel nhi paate hai inki vjha se.\nPlease take an action on priority.",dog,dog,stray,stray,harm_fear_or_nuisance_by_dogs,harm_fear_or_nuisance_by_dogs
2,"Tribal peoples are killing and eating innocent domestic pet animals like cats and dog in the area mentioned above. They are demanding more money to release the animal.\n\nPlace : Ambattur Telephone exchange road, tech Mahindra near signal",dog,unclear,pet,Not Applicable,Not Applicable,Not Applicable
3,"All Stray Dogs get bitten and murderd by people and also behave to much cruelty. No any Kidness With animals and also the people file munsipal dog cases againast animal rights. mostly peoples are bhanu nagar 2 and 3rd line areas. Address : Bhanu nagar,BRTS Road, Opposite Muthooth Fincorp, Vijayawda",dog,dog,stray,stray,harm_or_threat_to_dogs,harm_or_threat_to_dogs
4,"Dear sir/Madam, \n Need your help for killing innocent birds in my premises by local people by air-gun , already I sized the gun , need your help to catch the culprits. Talk to local IC , phone no :9147888670 but could not find any solution .Need your help for further action .\nThanking You \nArnab Nayak",bird,bird,Not Applicable,Not Applicable,Not Applicable,Not Applicable


In [166]:
handcoded.to_csv("handcoded_with_ai_labels_v4.csv", index=False)

##### **Testing accuracy**

In [167]:
#calculating the average times the two columns (ai & human) are equal
accuracy_animal = (handcoded["human_animal_type"] == handcoded["ai_animal_type"]).mean()
accuracy_dog = (handcoded["human_dog_type"] == handcoded["ai_dog_type"]).mean()
accuracy_complaint = (handcoded["human_complaint_type"] == handcoded["ai_complaint_type"]).mean()

print("animal_type accuracy:", accuracy_animal)
print("dog_type accuracy:", accuracy_dog)
print("complaint_type accuracy:", accuracy_complaint)

animal_type accuracy: 0.98
dog_type accuracy: 0.96
complaint_type accuracy: 0.96


In [168]:
#creating matching columns 
handcoded["animal_match"] = (
    handcoded["human_animal_type"] == handcoded["ai_animal_type"]
)

handcoded["dog_match"] = (
    handcoded["human_dog_type"] == handcoded["ai_dog_type"]
)

handcoded["complaint_match"] = (
    handcoded["human_complaint_type"] == handcoded["ai_complaint_type"]
)

In [169]:
#create a dataframe containing only rows where at least one column disagrees:

mismatches = handcoded[
    (~handcoded["animal_match"]) |
    (~handcoded["dog_match"]) |
    (~handcoded["complaint_match"])
]
len(mismatches)

4

In [170]:
pd.set_option("display.max_colwidth", None) #text wrap

animal_mismatches = handcoded[
    handcoded["human_animal_type"] != handcoded["ai_animal_type"]
]

animal_mismatches[
    [
        "complaint",
        "human_animal_type",
        "ai_animal_type"
    ]
]

,complaint,human_animal_type,ai_animal_type
2,"Tribal peoples are killing and eating innocent domestic pet animals like cats and dog in the area mentioned above. They are demanding more money to release the animal.\n\nPlace : Ambattur Telephone exchange road, tech Mahindra near signal",dog,unclear


In [171]:
handcoded.to_csv("handcoded_validated_ai_labels_final.csv", index=False)

In [173]:
# FINAL PROMPT VERSION
# Accuracy on 50 handcoded rows:
# animal_type: 0.98
# dog_type: 0.96
# complaint_type: 0.96
    
# You are classifying animal welfare complaint allegations.
# Classify based only on the text of the complaint. These are allegations, not verified facts.

# Broader rules:
# - Do not use outside knowledge. 
# - Do not infer beyond what is written. 
# - Choose exactly one value for each field.
# - If multiple issues appear, choose the category that best captures the dominant allegation.
# - Return only valid JSON.
# - Do not include explanations, markdown, confidence scores, or extra keys.

# Return this exact JSON structure:

# {{
# "animal_type": "",
# "dog_type": "",
# "complaint_type": ""
# }}

# Allowed values:

# animal_type:

# - dog 
# - cow
# - cat
# - horse
# - bird
# - sheep
# - unclear

# dog_type:

# - pet
# - stray
# - unclear
# - Not Applicable

# Allowed complaint_type values:
# - harm_or_threat_to_dogs
# - harm_or_threat_to_feeders
# - harm_or_threat_to_dogs_and_feeders
# - harm_due_to_fear
# - harm_fear_or_nuisance_by_dogs
# - unclear_or_other
# - Not Applicable


# 1. animal_type:
# - Use dog, cow, cat, horse, bird, sheep, unclear
# - Use unclear if the complaint does not clearly mention the animal and/or if the animal mentioned is not in the allowed values.

# 2. dog_type:
# - Applies only when animal_type is dog.
# - pet: use when the complaint is about owned and/or companion dog.
# - stray: use when the complaint is about stray dogs, street dogs, community dogs, free-roaming dogs, dogs in public spaces, or dogs cared for by feeders/caregivers. 
# If the complaint mentions dog catchers, municipal capture, ABC pickup, sterilisation pickup, release, relocation, or dogs being taken from a public area, treat the dog as stray unless the complaint clearly says it is a pet dog.
# Temporary care by residents, security guards, shelters, rescuers, or feeders does not make a dog a pet.
# - unclear: complaint mentions dogs but does not make clear whether pet or stray/community.
# - Not Applicable: use for all non-dog animal types.

# 3. complaint_type:

# 3a. Rules for complaint_type:

# - This field only applies when animal_type is "dog" and dog_type is "stray"
# - If animal_type is not "dog", complaint_type must be "Not Applicable".
# - If dog_type is "pet", "unclear_or_other", or "Not Applicable", complaint_type must be "Not Applicable".

# 3b. Stray/community dog complaint_type categories:

# * harm_or_threat_to_dogs:

# Use when the complaint alleges stray/community dogs were harmed, removed, neglected, exploited, or threatened with harm by human beings.

# This includes:
# - physical cruelty, including poisoning, beating, abuse, injury, killing, or throwing harmful liquids
# - threats to remove, relocate, poison, kill, injure, or otherwise harm stray/community dogs
# - illegal relocation or removal of stray/community dogs
# - dogs picked up and not returned to the same area
# - dogs reported missing
# - bad ABC, pound, or shelter conditions causing harm or neglect to dogs
# - vehicle collision or hit-and-run cases involving stray/community dogs, whether accidental or intentional


# * harm_or_threat_to_feeders:
# Use when the main allegation is that feeders, caregivers, or people supporting stray/community dogs were harassed, threatened, abused, attacked, or obstructed because they feed, care for, or speak for stray/community dogs.
# Use this only when the main allegation is harm or threat toward the feeder/caregiver, not direct harm or threat toward the dogs.
# Do not use this category merely because a feeder is mentioned. Harm or threat of harm should be clear.


# * harm_or_threat_to_dogs_and_feeders:

# Use only when the complaint clearly alleges both:

# 1. Harm, threat, cruelty, or obstruction directed toward stray/community dogs; and
# 2. Harm, threat, intimidation, harassment, abuse, or obstruction directed toward a person feeding, watering, rescuing, caring for, or advocating for stray/community dogs.

# Examples:
# - "Residents threatened to poison the dogs and abused the feeder."
# - "Residents beat the dogs and assaulted the person feeding them."
# - "People threw bricks at the dogs and threatened the complainant for giving them water."
# - "Residents destroyed the dogs' water bowls and intimidated the person who was caring for them."
# - "Don't feed the dog and relocate it to shelter"

# Do not use this category if only the dogs or only the feeder/caregiver is targeted. In those cases, use the corresponding single-target category.

# harm_fear_or_nuisance_by_dogs:
# Use when the complaint describes harm, fear, aggression, bites, barking, nuisance, danger, or requests to remove dogs because of public safety concerns.
# This includes complaints where dogs are described as attacking, biting, chasing, barking, creating nuisance, or causing fear among residents.

# harm_due_to_fear:
# Use when the complaint alleges that stray/community dogs or feeders/caregivers were harmed, threatened, abused, removed, or obstructed as a reaction to bites and aggression and public safetly concerns involving dogs.
# This category should be used when the complaint contains both:
# a fear/harm/dangerous concerns involving dogs, and
# harm, threat, abuse, removal, or retaliation directed toward dogs and/or feeders.

# Only classify as harm_fear_or_nuisance_by_dogs when the complaint explicitly states or clearly alleges that the dog was harmed or threatened because people feared the dog or considered it dangerous or a nuisance due to bites or aggresion. Do not infer this motivation simply because people wanted dogs removed, banned, relocated, or poisoned.

# Examples:
# “The dog bit a child, so residents poisoned the dog.”
# “After complaints about barking, residents beat the street dogs.”
# “Residents are threatening the feeder because the dogs allegedly chase people.”
# “Because of dog-bite fear, residents are demanding that the dogs be illegally relocated.”
# Do not use this category when the complaint only says dogs are biting, barking, chasing, or creating nuisance. In that case, use "harm_fear_or_nuisance_by_dogs".

# unclear_or_other:
# Use when the stray/community dog complaint is too vague or does not fit the categories above.
# E.g., "Stray dog was bitten / attacked by a pet dog", "Stray dog was attacked by a cow"
# """

### Running Classifier on National Dataset

In [175]:
df = pd.read_csv("national.csv")

In [177]:
df.head()

,sr_no,date_of_complaint,complaint_number,complainant_name_org,category_of_complaint,complaint,place_of_incident,violation_of_law,action_initiated,status,date_of_action_taken,state_id,page
0,1,12/04/2026,9-4/2026-27/3027/PCA,Unknown,-,"On 28.03.2026, a community puppy has been beaten to death by a person because it damaged his footwear. The locals rescued the pup and it was being treated at Humane Animal Society, Coimbatore but in vain. The pup had been suffering despite treatment and succumbed to its injuries on 10.04.2026.\r\n\r\nInjuring and causing death of an animal is a cognizable offense according to law but Tamilnadu Police Department repeatedly refuse to file FIR against the culprits in cases of Animal Cruelty. Hence it is requested that the necessary authorities may kindly be directed to officially register a case against the perpetrator and FIR be registered against the perpetrator.","Kakkan St, Rathinapuri, Tatabad, Coimbatore - 641 027.",-,-,-,21/04/2026,30,1
1,2,17/03/2026,9-4/2025-26/2917/PCA,Doctor Venkatesh,-,"Dr. Venkatesh of Lyka Pet Clinic, Dindigul, had performed ear cropping and tail bobbing aesthetic surgery on a doberman puppy belonging to Mr.Anand, appanaickenpalayam. phone 94430 15017. email : rakkianand22@gmail.com. The surgery was arranged by the breeder Anish of pannimadai, coimbatore, phone 99942 44737. The surgery was done not in a clinic but in the breeders breeding facility under unsanitary , non sterile conditions.",Appanaickenpalayam\r\nCoimbatore,-,-,-,21/04/2026,30,1
2,3,12/03/2026,9-4/2025-26/2900/PCA,security guards,-,"I am writing to formally bring to your attention a disturbing incident of animal cruelty that occurred inside the campus of Saveetha University, Chennai, Tamil Nadu.\r\n\r\nOn the university premises, specifically in the canteen area, a group of security guards reportedly captured and brutally killed a stray puppy. The act was carried out in a violent and inhumane manner within the campus grounds. This incident has caused significant distress among students and individuals who witnessed or later came to know about the event.\r\n\r\nThe puppy was a harmless stray animal that had been living around the campus. Instead of seeking assistance from authorized animal welfare organizations or following humane methods for handling stray animals, the individuals involved resorted to extreme cruelty, resulting in the death of the puppy. Such actions are deeply concerning and constitute a serious violation of animal protection laws in India.\r\n\r\nThis act appears to be a clear violation of the Prevention of Cruelty to Animals Act, 1960, which prohibits causing unnecessary pain or suffering to animals. Incidents like this not only demonstrate disregard for animal life but also create an unsafe and insensitive environment within an educational institution.\r\n\r\nI respectfully request the Animal Welfare Board of India to kindly investigate this matter and take appropriate action against the individuals responsible for this cruel act. It would also be helpful if the university administration is instructed to implement proper guidelines for the humane handling and protection of stray animals on campus.","Saveetha Engineering College\r\nSaveetha Nagar, Thandalam,\r\nChennai - Bangalore National Highway,\r\nKanchipuram",-,-,-,21/04/2026,30,1
3,4,04/02/2026,9-4/2025-26/2801/PCA,AOne Beef Stall,-,"Respected Sir / Madam,\r\n\r\nThis is bring to your kind notice, regarding my earlier complaint filed against illegal Slaughtering of COWS and BULLS. Complaint Number: 9-4/2025-26/2732/PCA.\r\n\r\nSo far only the Name Board: A-1 Beef Stall has been removed. No other action has been taken against the criminals.\r\n\r\nThe ILLEGAL SLAUGHTERING of Cows, Bulls and Calves is still happening in this illegal Slaughter-house on a Daily basis, which is functioning in a nameless Beef stall, built using blue coloured aluminium sheet.\r\n\r\nI am always available for any further assistance or clarification th

In [179]:
len(df)

1122

In [180]:
#creating categorisation columns 
df["ai_animal_type"] = ""
df["ai_dog_type"] = ""
df["ai_complaint_type"] = ""

In [181]:
# ensuring columns got created
df

,sr_no,date_of_complaint,complaint_number,complainant_name_org,category_of_complaint,complaint,place_of_incident,violation_of_law,action_initiated,status,date_of_action_taken,state_id,page,ai_animal_type,ai_dog_type,ai_complaint_type
0,1,12/04/2026,9-4/2026-27/3027/PCA,Unknown,-,"On 28.03.2026, a community puppy has been beaten to death by a person because it damaged his footwear. The locals rescued the pup and it was being treated at Humane Animal Society, Coimbatore but in vain. The pup had been suffering despite treatment and succumbed to its injuries on 10.04.2026.\r\n\r\nInjuring and causing death of an animal is a cognizable offense according to law but Tamilnadu Police Department repeatedly refuse to file FIR against the culprits in cases of Animal Cruelty. Hence it is requested that the necessary authorities may kindly be directed to officially register a case against the perpetrator and FIR be registered against the perpetrator.","Kakkan St, Rathinapuri, Tatabad, Coimbatore - 641 027.",-,-,-,21/04/2026,30,1,,,
1,2,17/03/2026,9-4/2025-26/2917/PCA,Doctor Venkatesh,-,"Dr. Venkatesh of Lyka Pet Clinic, Dindigul, had performed ear cropping and tail bobbing aesthetic surgery on a doberman puppy belonging to Mr.Anand, appanaickenpalayam. phone 94430 15017. email : rakkianand22@gmail.com. The surgery was arranged by the breeder Anish of pannimadai, coimbatore, phone 99942 44737. The surgery was done not in a clinic but in the breeders breeding facility under unsanitary , non sterile conditions.",Appanaickenpalayam\r\nCoimbatore,-,-,-,21/04/2026,30,1,,,
2,3,12/03/2026,9-4/2025-26/2900/PCA,security guards,-,"I am writing to formally bring to your attention a disturbing incident of animal cruelty that occurred inside the campus of Saveetha University, Chennai, Tamil Nadu.\r\n\r\nOn the university premises, specifically in the canteen area, a group of security guards reportedly captured and brutally killed a stray puppy. The act was carried out in a violent and inhumane manner within the campus grounds. This incident has caused significant distress among students and individuals who witnessed or later came to know about the event.\r\n\r\nThe puppy was a harmless stray animal that had been living around the campus. Instead of seeking assistance from authorized animal welfare organizations or following humane methods for handling stray animals, the individuals involved resorted to extreme cruelty, resulting in the death of the puppy. Such actions are deeply concerning and constitute a serious violation of animal protection laws in India.\r\n\r\nThis act appears to be a clear violation of the Prevention of Cruelty to Animals Act, 1960, which prohibits causing unnecessary pain or suffering to animals. Incidents like this not only demonstrate disregard for animal life but also create an unsafe and insensitive environment within an educational institution.\r\n\r\nI respectfully request the Animal Welfare Board of India to kindly investigate this matter and take appropriate action against the individuals responsible for this cruel act. It would also be helpful if the university administration is instructed to implement proper guidelines for the humane handling and protection of stray animals on campus.","Saveetha Engineering College\r\nSaveetha Nagar, Thandalam,\r\nChennai - Bangalore National Highway,\r\nKanchipuram",-,-,-,21/04/2026,30,1,,,
3,4,04/02/2026,9-4/2025-26/2801/PCA,AOne Beef Stall,-,"Respected Sir / Madam,\r\n\r\nThis is bring to your kind notice, regarding my earlier complaint filed against illegal Slaughtering of COWS and BULLS. Complaint Number: 9-4/2025-26/2732/PCA.\r\n\r\nSo far only the Name Board: A-1 Beef Stall has been removed. No other action has been taken against the criminals.\r\n\r\nThe ILLEGAL SLAUGHTERING of Cows, Bulls and Calves is still happening in this illegal Slaughter-house on a Daily basis, which is functioning in a nameless Beef stall, built using blue coloured aluminium sheet.\r\n\r\nI am always av

In [183]:
model = "openai/gpt-4o-mini"

for idx, row in df.iterrows():
    complaint = row["complaint"]

    try:
        ai_answer = ask_ai_to_classify(complaint, model=model)
        parsed = json.loads(ai_answer)

        df.loc[idx, "ai_animal_type"] = parsed["animal_type"]
        df.loc[idx, "ai_dog_type"] = parsed["dog_type"]
        df.loc[idx, "ai_complaint_type"] = parsed["complaint_type"]

        print(f"Done row {idx}")

    except Exception as e:
        print(f"Error on row {idx}: {e}")
        print(ai_answer if "ai_answer" in locals() else "No AI answer")

    time.sleep(1)

Done row 0
Done row 1
Done row 2
Done row 3
Done row 4
Done row 5
Done row 6
Done row 7
Done row 8
Done row 9
Done row 10
Done row 11
Done row 12
Done row 13
Done row 14
Done row 15
Done row 16
Done row 17
Done row 18
Done row 19
Done row 20
Done row 21
Done row 22
Done row 23
Done row 24
Done row 25
Done row 26
Done row 27
Done row 28
Done row 29
Done row 30
Done row 31
Done row 32
Done row 33
Done row 34
Done row 35
Done row 36
Done row 37
Done row 38
Done row 39
Done row 40
Done row 41
Done row 42
Done row 43
Done row 44
Done row 45
Done row 46
Done row 47
Done row 48
Done row 49
Done row 50
Done row 51
Done row 52
Done row 53
Done row 54
Done row 55
Done row 56
Done row 57
Done row 58
Done row 59
Done row 60
Done row 61
Done row 62
Done row 63
Done row 64
Done row 65
Done row 66
Done row 67
Done row 68
Done row 69
Done row 70
Done row 71
Done row 72
Done row 73
Done row 74
Done row 75
Done row 76
Done row 77
Done row 78
Done row 79
Done row 80
Done row 81
Done row 82
Done row 83
Do

In [184]:
df.to_csv("national_with_ai_labels.csv", index=False)